In [4]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('filtered_daily_data.csv')

In [8]:
import pandas as pd
import numpy as np

def evaluate_daily_portfolio(df_daily, weights_csv_path, tc_rate=0.001):
    print("Preparing datasets...")
    
    # 1. Load weights and parse dates
    df_weights = pd.read_csv(weights_csv_path)
    # Handles the '10/31/21' format from your screenshot
    df_weights['date'] = pd.to_datetime(df_weights['date']) 
    
    # Ensure daily dates are datetime (handles '1990-10-01' format)
    df_daily = df_daily.copy()
    df_daily['date'] = pd.to_datetime(df_daily['date'])
    
    # 2. Map Weights Forward
    # e.g., 10/31/21 weights are mapped to a '2021-11' trading_month tag
    df_weights['trading_month'] = (df_weights['date'] + pd.offsets.MonthBegin(1)).dt.to_period('M')
    df_daily['trading_month'] = df_daily['date'].dt.to_period('M')
    
    print("Merging data and calculating daily drift...")
    # 3. Merge weights into daily returns
    df = pd.merge(
        df_daily, 
        df_weights[['trading_month', 'permno', 'weight']], 
        on=['trading_month', 'permno'], 
        how='inner'
    )
    
    df.rename(columns={'weight': 'target_weight'}, inplace=True)
    df.sort_values(['permno', 'date'], inplace=True)
    
    # 4. Calculate Daily Drifted Weights and Gross Returns
    # Using 'ret_excess' as shown in your daily data screenshot
    df['1+r'] = 1 + df['ret_excess'].fillna(0)
    df['cum_ret'] = df.groupby(['trading_month', 'permno'])['1+r'].cumprod()
    
    # V_t is the position value at the end of day t
    df['V_t'] = df['target_weight'] * df['cum_ret']
    
    # V_t_minus_1 is the position value at the start of day t
    df['V_t_minus_1'] = df.groupby(['trading_month', 'permno'])['V_t'].shift(1)
    df['V_t_minus_1'] = df['V_t_minus_1'].fillna(df['target_weight'])
    
    df['dollar_ret'] = df['V_t_minus_1'] * df['ret_excess']
    
    # Aggregate to daily portfolio level
    port_daily = df.groupby('date').agg(
        total_dollar_ret=('dollar_ret', 'sum'),
        total_port_value=('V_t_minus_1', 'sum')
    ).reset_index()
    
    port_daily['gross_ret'] = port_daily['total_dollar_ret'] / port_daily['total_port_value']
    
    print("Calculating month-end turnover and transaction costs...")
    # 5. Calculate Turnover & TC
    tc_records = []
    
    # Initial Portfolio Setup TC (Month 1)
    initial_month = df_weights['trading_month'].min()
    initial_weights = df_weights[df_weights['trading_month'] == initial_month]
    initial_tc = initial_weights['weight'].abs().sum() * tc_rate
    tc_records.append({'trading_month': initial_month, 'tc': initial_tc})
    
    # Rebalancing TC (Month M to Month M+1)
    # Get drifted weight on the last trading day of each month
    last_days = df.groupby(['trading_month', 'permno'])['date'].max().reset_index()
    eom_df = pd.merge(df, last_days, on=['trading_month', 'permno', 'date'])
    
    eom_port_val = eom_df.groupby('trading_month')['V_t'].sum().reset_index(name='port_V_end')
    eom_df = pd.merge(eom_df, eom_port_val, on='trading_month')
    eom_df['drifted_weight'] = eom_df['V_t'] / eom_df['port_V_end']
    
    # Align month M drifted weights with month M+1 target weights
    eom_df['next_trading_month'] = eom_df['trading_month'] + 1
    next_weights = df_weights[['trading_month', 'permno', 'weight']].copy()
    next_weights.rename(columns={'weight': 'target_weight'}, inplace=True)
    
    turnover_df = pd.merge(
        eom_df[['next_trading_month', 'permno', 'drifted_weight']], 
        next_weights,
        left_on=['next_trading_month', 'permno'],
        right_on=['trading_month', 'permno'],
        how='outer'
    )
    
    turnover_df['drifted_weight'] = turnover_df['drifted_weight'].fillna(0)
    turnover_df['target_weight'] = turnover_df['target_weight'].fillna(0)
    turnover_df['turnover'] = (turnover_df['target_weight'] - turnover_df['drifted_weight']).abs()
    
    # Sum turnover per month and multiply by 10 bps
    monthly_tc = turnover_df.groupby('next_trading_month')['turnover'].sum() * tc_rate
    for tm, tc in monthly_tc.items():
        tc_records.append({'trading_month': tm, 'tc': tc})
        
    tc_df = pd.DataFrame(tc_records)
    
    # 6. Apply TC on the first trading day of the month
    port_daily['trading_month'] = port_daily['date'].dt.to_period('M')
    first_days = port_daily.groupby('trading_month')['date'].min().reset_index()
    first_days = pd.merge(first_days, tc_df, on='trading_month', how='left')
    
    port_daily = pd.merge(port_daily, first_days[['date', 'tc']], on='date', how='left')
    port_daily['tc'] = port_daily['tc'].fillna(0)
    
    port_daily['net_ret'] = port_daily['gross_ret'] - port_daily['tc']
    port_daily['cum_ret'] = (1 + port_daily['net_ret']).cumprod() - 1
    
    # Display Stats
    mean_ret = port_daily['net_ret'].mean()
    std_ret = port_daily['net_ret'].std()
    ann_sharpe = (mean_ret / std_ret) * np.sqrt(252) if std_ret > 0 else 0
    
    print("\n" + "="*45)
    print("DAILY PORTFOLIO METRICS (Net of 10bps TC)")
    print("="*45)
    print(f"Annualized Sharpe Ratio : {ann_sharpe:.4f}")
    print(f"Total Cumulative Return : {port_daily['cum_ret'].iloc[-1]*100:.2f}%")
    print(f"Average Daily Return    : {mean_ret*10000:.2f} bps")
    print(f"Average Rebalance Cost  : {tc_df['tc'].mean()*10000:.2f} bps per month")
    print("="*45)
    
    return port_daily

In [38]:
# Execute the function
# ==========================================
# Execution
# ==========================================
# Assuming your daily data is already loaded:
# df = pd.read_csv('filtered_daily_data.csv')

daily_results = evaluate_daily_portfolio(
    df_daily=df, 
    weights_csv_path='../GPT 4o/Portfolio weights/nls_weights_msr.csv', # Replace with your actual weights filename
    tc_rate=0.001
)

Preparing datasets...
Merging data and calculating daily drift...
Calculating month-end turnover and transaction costs...

DAILY PORTFOLIO METRICS (Net of 10bps TC)
Annualized Sharpe Ratio : 3.3842
Total Cumulative Return : 30.14%
Average Daily Return    : 21.61 bps
Average Rebalance Cost  : 18.13 bps per month


In [44]:
daily_results['net_ret'].to_csv('agentic_ai_daily_returns.csv', index=False)

In [47]:
import pandas as pd
import numpy as np

def evaluate_daily_rebalanced_ew(df_daily, tc_rate=0.001):
    print("Preparing daily Equal-Weighted targets...")
    
    df = df_daily.copy()
    df['date'] = pd.to_datetime(df['date'])
    
    # 1. Calculate Daily Target Weights (1/N) and Gross Returns
    # N is the number of active stocks on that specific day
    daily_counts = df.groupby('date')['permno'].nunique().reset_index(name='N')
    daily_counts['target_w'] = 1.0 / daily_counts['N']
    
    df = pd.merge(df, daily_counts[['date', 'target_w']], on='date', how='left')
    
    # Gross daily return is simply the equal-weighted average of returns that day
    port_daily = df.groupby('date')['ret_excess'].mean().reset_index()
    port_daily.rename(columns={'ret_excess': 'gross_ret'}, inplace=True)
    
    # Merge gross portfolio return back to calculate precise end-of-day drift
    df = pd.merge(df, port_daily, on='date', how='left')
    
    print("Calculating daily drift and turnover...")
    # 2. Calculate End-of-Day (EOD) Drifted Weights
    # If a stock outperforms the daily average, its weight drifts slightly above 1/N by the close
    df['1+r'] = 1 + df['ret_excess'].fillna(0)
    df['1+R_port'] = 1 + df['gross_ret']
    df['drift_w'] = df['target_w'] * (df['1+r'] / df['1+R_port'])
    
    # 3. Map EOD weights to the next trading day to calculate turnover
    dates = sorted(df['date'].unique())
    next_date_map = dict(zip(dates[:-1], dates[1:]))
    df['next_date'] = df['date'].map(next_date_map)
    
    # Separate End-of-Day (yesterday) and Start-of-Day (today) frames
    eod_df = df.dropna(subset=['next_date'])[['next_date', 'permno', 'drift_w']]
    sod_df = df[['date', 'permno', 'target_w']]
    
    # Merge yesterday's closing weights with today's target weights
    turnover_df = pd.merge(
        sod_df, 
        eod_df, 
        left_on=['date', 'permno'], 
        right_on=['next_date', 'permno'], 
        how='outer'
    )
    
    # Fill missing with 0 (for stocks entering or exiting the universe)
    turnover_df['target_w'] = turnover_df['target_w'].fillna(0)
    turnover_df['drift_w'] = turnover_df['drift_w'].fillna(0)
    
    # Daily Turnover = Sum of absolute weight changes required to snap back to 1/N
    turnover_df['turnover'] = (turnover_df['target_w'] - turnover_df['drift_w']).abs()
    
    # Sum turnover across all stocks for each day
    daily_tc = turnover_df.groupby('date')['turnover'].sum() * tc_rate
    daily_tc_df = daily_tc.reset_index(name='tc')
    
    # 4. Handle Initial Setup TC (Day 1)
    # On the very first day, you pay TC on 100% of the portfolio to establish the positions
    first_date = dates[0]
    daily_tc_df.loc[daily_tc_df['date'] == first_date, 'tc'] = 1.0 * tc_rate
    
    print("Finalizing net returns...")
    # 5. Apply Daily TC to Gross Returns
    port_daily = pd.merge(port_daily, daily_tc_df, on='date', how='left')
    port_daily['tc'] = port_daily['tc'].fillna(0)
    
    port_daily['net_ret'] = port_daily['gross_ret'] - port_daily['tc']
    port_daily['cum_ret'] = (1 + port_daily['net_ret']).cumprod() - 1
    
    # Display Stats
    mean_ret = port_daily['net_ret'].mean()
    std_ret = port_daily['net_ret'].std()
    ann_sharpe = (mean_ret / std_ret) * np.sqrt(252) if std_ret > 0 else 0
    
    print("\n" + "="*45)
    print("DAILY-REBALANCED EW BENCHMARK (Net of 10bps TC)")
    print("="*45)
    print(f"Annualized Sharpe Ratio : {ann_sharpe:.4f}")
    print(f"Total Cumulative Return : {port_daily['cum_ret'].iloc[-1]*100:.2f}%")
    print(f"Average Daily Return    : {mean_ret*10000:.2f} bps")
    print(f"Average Daily TC Paid   : {port_daily['tc'].mean()*10000:.2f} bps")
    print("="*45)
    
    return port_daily

In [48]:
df['date'] = pd.to_datetime(df['date'])
df_daily_filtered = df[(df['date'] > '2023-11-30') & (df['date'] <= '2024-05-31')].copy()

daily_ew_results = evaluate_daily_rebalanced_ew(df_daily=df_daily_filtered, tc_rate=0.001)

Preparing daily Equal-Weighted targets...
Calculating daily drift and turnover...
Finalizing net returns...

DAILY-REBALANCED EW BENCHMARK (Net of 10bps TC)
Annualized Sharpe Ratio : 1.7088
Total Cumulative Return : 10.01%
Average Daily Return    : 7.90 bps
Average Daily TC Paid   : 0.18 bps


In [51]:
daily_ew_results['net_ret'].to_csv('ew_daily_returns.csv', index=False)